# Figures of the enrichment analysis

Reads only `pd-lcm-rf-enrichment`; nothing is recomputed. Print size, 183 mm wide.

* **Figure 8** - the 30 Boruta genes on the dopamine-neuron lineage axis (Kamath et al. 2022)
* **Figure 9** - pathway over-representation and pathway partners, every pathway named in full
* **Figure S** - a one-page summary of all 30 panel genes

In [ ]:
import zipfile, re, xml.etree.ElementTree as ET
import pandas as pd
def read_strict_xlsx(path):
    """Every sheet of an .xlsx workbook as a DataFrame - handles 'strict' OOXML, which openpyxl cannot open."""
    zx = zipfile.ZipFile(path)
    wb = ET.fromstring(zx.read("xl/workbook.xml"))
    M = wb.tag.split("}")[0].strip("{")                                  # main namespace, strict or transitional
    rels = ET.fromstring(zx.read("xl/_rels/workbook.xml.rels"))
    target = {r.get("Id"): r.get("Target") for r in rels}
    RID = [k for k in wb.find(f"{{{M}}}sheets")[0].attrib if k.endswith("}id")][0]
    ss = []
    if "xl/sharedStrings.xml" in zx.namelist():
        for si in ET.fromstring(zx.read("xl/sharedStrings.xml")).findall(f"{{{M}}}si"):
            ss.append("".join(t.text or "" for t in si.iter(f"{{{M}}}t")))
    col = lambda ref: sum((ord(ch) - 64) * 26 ** i for i, ch in enumerate(reversed(re.match(r"[A-Z]+", ref).group())))
    out = {}
    for sh in wb.find(f"{{{M}}}sheets"):
        f = "xl/" + target[sh.get(RID)].lstrip("/").replace("xl/", "")
        rows = []
        for r in ET.fromstring(zx.read(f)).iter(f"{{{M}}}row"):
            vals = {}
            for c in r.findall(f"{{{M}}}c"):
                v = c.findtext(f"{{{M}}}v")
                if c.get("t") == "s" and v is not None:
                    v = ss[int(v)]
                elif c.get("t") == "inlineStr":
                    v = "".join(t.text or "" for t in c.iter(f"{{{M}}}t"))
                vals[col(c.get("r")) - 1] = v
            rows.append([vals.get(i) for i in range(max(vals) + 1)] if vals else [])
        out[sh.get("name")] = pd.DataFrame(rows)
    return out


In [ ]:
import os, re, json, glob, sys, subprocess, textwrap
from pathlib import Path
import numpy as np, pandas as pd
ON_KAGGLE = Path("/kaggle/input").exists()
OUT = Path("/kaggle/working") if ON_KAGGLE else Path(os.environ.get("FIG_OUT", "."))
def fin(name):
    if not ON_KAGGLE:
        return Path(os.environ["ENR_IN"]) / name
    hits = sorted((h for h in glob.glob(f"/kaggle/input/**/{name}", recursive=True) if "rf-enrichment" in h), key=len)
    if not hits:
        raise FileNotFoundError(name)
    return hits[0]
try:
    from adjustText import adjust_text
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "adjustText"], check=True)
    from adjustText import adjust_text

BGL = pd.read_csv(fin("enrich_background_lineage.csv"))
PLN = pd.read_csv(fin("enrich_panel_lineage.csv"))
SBT = pd.read_csv(fin("enrich_subtypes.csv"))
ORT = pd.read_csv(fin("enrich_ora.csv"))
NBT = pd.read_csv(fin("enrich_pathway_neighbours.csv"))
GTB = pd.read_csv(fin("enrich_gene_table.csv"))
SMY = json.load(open(fin("enrich_summary.json")))
SU = SMY["subtypes"]
NICE = dict(zip(GTB.symbol.str.upper(), GTB.symbol))                     # upper-case symbol -> official capitalisation
nice = lambda s: NICE.get(str(s).upper(), s)
PLN["name"] = PLN.symbol.map(nice)
PLN = PLN.merge(GTB[["gene", "shap_rank"]], on="gene", how="left")
DOWN = PLN[PLN.direction == "down"].sort_values("lineage").name.tolist()
UP = PLN[PLN.direction == "up"].sort_values("lineage").name.tolist()
DIRN = dict(zip(PLN.name, PLN.direction))

# Kamath et al. 2022, Supplementary Table 8: marker z of every panel gene in every dopamine-neuron subtype
T8 = read_strict_xlsx(fin("kamath_2022_supplementary.xlsx"))["Supplementary_Table_8"]
T8.columns = [str(c) if c is not None else f"c{i}" for i, c in enumerate(T8.iloc[0])]; T8 = T8.iloc[1:]
T8["z"] = pd.to_numeric(T8["z"], errors="coerce"); T8["symbol"] = T8.primerid.astype(str).str.upper()
SUBTYPES = ["SOX6_AGTR1", "SOX6_PART1", "SOX6_DDT", "SOX6_GFRA2", "CALB1_CALCR", "CALB1_CRYM_CCDC68", "CALB1_GEM",
            "CALB1_PPP1R17", "CALB1_RBP4", "CALB1_TRHR"]
ZP = (T8.pivot_table(index="symbol", columns="DA_subtype", values="z", aggfunc="first")
        .reindex(index=[g.upper() for g in DOWN + UP], columns=SUBTYPES).fillna(0.0))
ZP.index = DOWN + UP

LIBTAG = {"GO:BP": "GO BP", "GO:CC": "GO CC", "GO:MF": "GO MF", "Reactome": "Reactome", "KEGG": "KEGG", "WikiPathways": "WikiPW"}
def term_name(t):
    """Full pathway name, identifiers removed, sentence case with acronyms kept."""
    t = re.sub(r"\s*\((GO:\d+)\)$", "", t); t = re.sub(r"\s+R-HSA-\d+$", "", t); t = re.sub(r"\s+WP\d+$", "", t)
    words = t.split(" ")
    out = [words[0]] + [w if (w.isupper() and len(w) > 1) or re.search(r"\d", w) else w.lower() for w in words[1:]]
    t = " ".join(out)
    for a in ("dna", "rna", "gpcrs", "gpcr", "hsf1", "h4/h2a", "h2a", "h4"):
        t = re.sub(rf"(?i)\b{re.escape(a)}\b", a.upper() if a not in ("gpcrs",) else "GPCRs", t)
    return t
wrap = lambda t, w: textwrap.wrap(t, w)
print(f"panel: {len(DOWN)} down, {len(UP)} up; Kamath markers for the panel: {int((ZP > 0).sum().sum())} gene-subtype pairs")


In [ ]:
import matplotlib
try:
    get_ipython(); IN_NB = True
except NameError:
    IN_NB = False; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.patches import Rectangle, Polygon, FancyBboxPatch
from matplotlib.path import Path as MPath
from matplotlib.patches import PathPatch

INK, MUTED, RULE = "#1B1D20", "#5E656D", "#C9CED4"
PD_C, CT_C = "#7A2533", "#6F829A"
CORE_C, PANEL_C, NEURO_C = "#23324A", "#8E1B2E", "#9AA1A9"
SETS = {"both": ("#8E1B2E", "#5E0F1C", "Boruta & DEG"), "boruta": ("#2E5A87", "#1B3A5C", "Boruta only")}
EFFECT = LinearSegmentedColormap.from_list("effect", ["#1E3350", "#4C6583", "#9AAABB", "#EEECE7", "#C3A09E", "#8C4B52", "#551C28"])
FONT_STACK = ["Helvetica Neue", "Helvetica", "Arial", "Liberation Sans", "Nimbus Sans", "FreeSans", "DejaVu Sans"]
plt.rcParams.update({"font.family": "sans-serif", "font.sans-serif": FONT_STACK, "font.size": 6.3,
                     "axes.linewidth": 0.5, "axes.edgecolor": "#30343A", "axes.labelcolor": INK, "text.color": INK,
                     "axes.spines.top": False, "axes.spines.right": False, "xtick.labelsize": 5.9, "ytick.labelsize": 5.9,
                     "xtick.major.width": 0.5, "ytick.major.width": 0.5, "xtick.major.size": 2.1, "ytick.major.size": 2.1,
                     "xtick.major.pad": 1.7, "ytick.major.pad": 1.7, "xtick.color": "#30343A", "ytick.color": "#30343A",
                     "pdf.fonttype": 42, "ps.fonttype": 42, "figure.dpi": 150, "savefig.facecolor": "white",
                     "figure.facecolor": "white", "mathtext.fontset": "custom", "mathtext.rm": "sans", "mathtext.it": "sans:italic"})

class Canvas:
    """A print-size figure drawn on a millimetre grid."""
    def __init__(self, w, h):
        self.W, self.H = w, h
        self.fig = plt.figure(figsize=(w / 25.4, h / 25.4))
        self.M = self.fig.add_axes([0, 0, 1, 1]); self.M.set_xlim(0, w); self.M.set_ylim(0, h); self.M.axis("off")
    def ax(self, x, y, w, h):
        return self.fig.add_axes([x / self.W, y / self.H, w / self.W, h / self.H])
    def letter(self, x, y, L, title):
        self.M.text(x, y, L, ha="left", va="baseline", fontsize=9, fontweight="bold", color=INK)
        self.M.text(x + 4.2, y, title, ha="left", va="baseline", fontsize=7.2, color=INK)
    def rule(self, x0, x1, y, lw=0.4, color=RULE):
        self.M.plot([x0, x1], [y, y], color=color, lw=lw, solid_capstyle="butt")
    def text(self, x, y, s, **kw):
        kw.setdefault("va", "center"); kw.setdefault("fontsize", 6.0)
        return self.M.text(x, y, s, **kw)
    def save(self, stem):
        for ext in ("pdf", "png"):
            self.fig.savefig(OUT / f"{stem}.{ext}", dpi=600 if ext == "png" else None)
        print("saved", stem)
        plt.show() if IN_NB else plt.close(self.fig)
f2 = lambda v: f"{v:.2f}"
def pfmt(p): return f"= {p:.3f}" if p >= 0.001 else "< 0.001"   # "P = 0.003", "P < 0.0001"


## Figure 8 - dopamine-neuron subtypes (183 x 180 mm)

In [ ]:
from matplotlib.colors import LinearSegmentedColormap, Normalize
UPC, DNC = PANEL_C, CORE_C
C = Canvas(183.0, 180.0); M = C.M
LT = 2.0
tr = lambda v: np.sign(v) * np.log10(1 + np.abs(v) / LT)       # symmetric log for the lineage score
pfmt2 = lambda p: f"P = {p:.4f}" if p >= 0.0001 else "P < 0.0001"

# ======================= a: PD effect against lineage, every gene =======================
C.letter(2.0, 176.0, "a", "PD effect and dopamine-neuron lineage of every measured gene")
AX0, AY0, AW, AH = 17.0, 108.0, 90.0, 51.0
ax = C.ax(AX0, AY0, AW, AH)
xlo, xhi = tr(-22.0), tr(19.0); ylo, yhi = -1.45, 1.45
ax.set_xlim(xlo, xhi); ax.set_ylim(ylo, yhi)
p05, p95 = tr(SU["background_p05"]), tr(SU["background_p95"])
ax.add_patch(Rectangle((xlo, ylo), p05 - xlo, -ylo, facecolor=DNC, alpha=0.06, lw=0, zorder=0))
ax.add_patch(Rectangle((p95, 0), xhi - p95, yhi, facecolor=UPC, alpha=0.06, lw=0, zorder=0))
ax.axhline(0, color="#AEB4BB", lw=0.5, zorder=1); ax.axvline(0, color="#AEB4BB", lw=0.5, zorder=1)
for v in (p05, p95):
    ax.axvline(v, color="#C9CED4", lw=0.5, ls=(0, (2, 2)), zorder=1)
ax.scatter(tr(BGL.lineage), BGL.g, s=1.1, color="#9AA1A9", alpha=0.35, lw=0, zorder=2, rasterized=True)
size = lambda r: 16 + (31 - r) * 1.25
for d, col in (("down", DNC), ("up", UPC)):
    s_ = PLN[PLN.direction == d]
    ax.scatter(tr(s_.lineage), s_.g, s=size(s_.shap_rank), color=col, edgecolor="white", linewidth=0.5, zorder=4)
TK = [-16, -8, -4, -2, 0, 2, 4, 8, 16]
ax.set_xticks([tr(v) for v in TK]); ax.set_xticklabels([f"{v:+d}" if v else "0" for v in TK])
ax.set_yticks([-1, -0.5, 0, 0.5, 1]); ax.set_yticklabels(["-1", "-0.5", "0", "0.5", "1"])
ax.set_xlabel("Dopamine-neuron lineage score (CALB1 minus SOX6 subtype markers, symmetric log)", fontsize=6.0, labelpad=2)
ax.set_ylabel("PD effect in discovery neurons (Hedges' g)", fontsize=6.0, labelpad=2)
texts = []
for r in PLN.itertuples():
    texts.append(ax.text(tr(r.lineage), r.g, r.name, fontsize=5.6, fontstyle="italic", color=DNC if r.direction == "down" else UPC,
                         ha="center", va="center", zorder=6))
adjust_text(texts, x=list(tr(PLN.lineage)), y=list(PLN.g), ax=ax, expand=(1.25, 1.5), force_text=(0.4, 0.6),
            force_static=(0.4, 0.6), arrowprops=dict(arrowstyle="-", color="#8C9299", lw=0.35, shrinkA=0, shrinkB=2), max_move=None)
ax.text(xlo + 0.02, ylo + 0.05, "Lower in PD\nSOX6 lineage", fontsize=5.6, color=DNC, ha="left", va="bottom", fontweight="bold",
        linespacing=1.0, zorder=5)
ax.text(xhi - 0.02, 0.06, "Higher in PD\nCALB1 lineage", fontsize=5.6, color=UPC, ha="right", va="bottom", fontweight="bold",
        linespacing=1.0, zorder=5)
# marginal distributions: all genes in grey, panel genes as ticks
axT = C.ax(AX0, AY0 + AH + 0.8, AW, 7.0); axT.set_xlim(xlo, xhi); axT.axis("off")
hb, eb = np.histogram(tr(BGL.lineage), bins=np.linspace(xlo, xhi, 70))
axT.fill_between(np.repeat(eb, 2)[1:-1], 0, np.repeat(np.sqrt(hb), 2), color="#C9CED4", lw=0)
axT.set_ylim(-0.6 * np.sqrt(hb).max(), np.sqrt(hb).max() * 1.05)
for r in PLN.itertuples():
    axT.plot([tr(r.lineage)] * 2, [-0.55 * np.sqrt(hb).max(), -0.08 * np.sqrt(hb).max()], color=DNC if r.direction == "down" else UPC, lw=0.7)
axR = C.ax(AX0 + AW + 0.8, AY0, 7.0, AH); axR.set_ylim(ylo, yhi); axR.axis("off")
hg, eg = np.histogram(BGL.g, bins=np.linspace(ylo, yhi, 60))
axR.fill_betweenx(np.repeat(eg, 2)[1:-1], 0, np.repeat(np.sqrt(hg), 2), color="#C9CED4", lw=0)
axR.set_xlim(-0.6 * np.sqrt(hg).max(), np.sqrt(hg).max() * 1.05)
for r in PLN.itertuples():
    axR.plot([-0.55 * np.sqrt(hg).max(), -0.08 * np.sqrt(hg).max()], [r.g] * 2, color=DNC if r.direction == "down" else UPC, lw=0.7)
C.text(AX0, 171.0, f"Grey: all {len(BGL):,} measured genes.  Coloured: the 30 Boruta genes; dot area: SHAP rank",
       ha="left", fontsize=5.6, color=MUTED)
for k, rk in enumerate((1, 10, 20, 30)):
    M.scatter([AX0 + 81.0 + k * 4.2], [171.0], s=size(rk), color="#6B7178", edgecolor="white", linewidth=0.4)
    C.text(AX0 + 81.0 + k * 4.2, 168.4, str(rk), ha="center", fontsize=4.8, color=MUTED)
KY = 97.0
C.text(AX0, KY, f"Down vs up genes on the lineage axis: {pfmt2(SU['up_vs_down'])} (Mann-Whitney); against genes equally changed in PD: "
       f"{pfmt2(SU['p_vs_DE_matched'])}; dashed lines: 5th and 95th percentiles of all genes", ha="left", fontsize=5.5, color=INK)

# ======================= b: subtype marker overlap, down (left) and up (right) =======================
BX = 124.0
C.letter(BX, 176.0, "b", "Marker overlap by subtype")
L0, L1, R0, R1, LABX = 128.0, 144.0, 165.0, 181.0, 154.5
XMAX = 2.2
sc = lambda v, left: (L1 - (L1 - L0) * v / XMAX) if left else (R0 + (R1 - R0) * v / XMAX)
C.text((L0 + L1) / 2, 167.8, "Lower in PD (12)", ha="center", fontsize=5.8, color=DNC, fontweight="bold")
C.text((R0 + R1) / 2, 167.8, "Higher in PD (18)", ha="center", fontsize=5.8, color=UPC, fontweight="bold")
sbt = SBT.set_index(["subtype", "list"])
ROWY = {}
yy = 162.5
for i, st in enumerate(SUBTYPES):
    if i == 4:
        yy -= 3.2
    if i in (0, 4):
        grp = "SOX6 lineage (vulnerable)" if i == 0 else "CALB1 lineage (resilient)"
        C.text(LABX, yy + 0.6, grp, ha="center", fontsize=5.5, color=DNC if i == 0 else UPC, fontstyle="italic")
        yy -= 3.6
    ROWY[st] = yy; yy -= 4.3
top_b, bot_b = ROWY[SUBTYPES[0]] + 2.4, ROWY[SUBTYPES[-1]] - 2.4
thr = -np.log10(0.05)
for sts in (SUBTYPES[:4], SUBTYPES[4:]):
    ya, yb = ROWY[sts[0]] + 2.3, ROWY[sts[-1]] - 2.3
    if sts[-1] == SUBTYPES[-1]:
        yb = bot_b
    for x in (L1, R0):
        M.plot([x, x], [yb, ya], color="#8C9299", lw=0.5)
    for left in (True, False):
        xv = sc(thr, left)
        M.plot([xv, xv], [yb, ya], color="#8C9299", lw=0.5, ls=(0, (2, 1.6)))
for st in SUBTYPES:
    yy = ROWY[st]
    C.text(LABX, yy, st.replace("_", " "), ha="center", fontsize=5.2, color=INK, fontweight="bold" if st == "SOX6_AGTR1" else "normal")
    for d, left, col in (("down", True, DNC), ("up", False, UPC)):
        r = sbt.loc[(st, d)]
        v = min(-np.log10(r.q_BH), XMAX)
        if r.hits:
            x0, x1 = sorted((sc(0, left), sc(v, left)))
            M.add_patch(Rectangle((x0, yy - 1.3), max(x1 - x0, 0.25), 2.6, facecolor=col if r.q_BH < 0.05 else "white",
                                  edgecolor=col, lw=0.6, zorder=3))
            xt = sc(v, left) + (-0.8 if left else 0.8)
            C.text(xt, yy, str(int(r.hits)), ha="right" if left else "left", fontsize=5.2, color=col, zorder=4)
for left in (True, False):
    for v in (0, 1, 2):
        xv = sc(v, left)
        M.plot([xv, xv], [bot_b - 0.2, bot_b - 1.1], color=INK, lw=0.5)
        C.text(xv, bot_b - 2.6, str(v), ha="center", fontsize=5.4)
    M.plot([sc(0, left), sc(XMAX, left)], [bot_b - 0.2, bot_b - 0.2], color=INK, lw=0.5)
C.text(LABX, bot_b - 5.6, "$-\\log_{10}$ FDR  (dashed: FDR 0.05)", ha="center", fontsize=5.6)
C.text(LABX, bot_b - 8.6, "numbers: panel genes among the subtype's 200 top markers", ha="center", fontsize=5.2, color=MUTED)

# ======================= c: marker strength, gene by subtype =======================
C.rule(2.0, 181.0, 93.2, lw=0.35, color="#8C9299")
C.letter(2.0, 88.8, "c", "Marker strength of each panel gene in each dopamine-neuron subtype (Kamath et al. 2022)")
genes = DOWN + UP
HX0, HX1 = 40.0, 178.0
cw = (HX1 - HX0) / len(genes)
gx = {g: HX0 + (i + 0.5) * cw + (1.2 if g in UP else 0) for i, g in enumerate(genes)}
HX1 += 1.2
# PD effect bars above the heatmap
EB0, EH = 70.0, 9.0
gmap = dict(zip(PLN.name, PLN.g)); gmax = 1.3
M.plot([HX0 - 0.5, HX1 + 0.5], [EB0 + EH / 2] * 2, color="#8C9299", lw=0.4)
for g in genes:
    v = gmap[g]; h = EH / 2 * v / gmax
    M.add_patch(Rectangle((gx[g] - cw * 0.32, EB0 + EH / 2 + min(0, h)), cw * 0.64, abs(h), facecolor=DNC if v < 0 else UPC, lw=0))
C.text(HX0 - 2.0, EB0 + EH / 2, "PD effect (g)", ha="right", fontsize=5.6)
for v, lab in ((gmax, "+1.3"), (-gmax, "-1.3")):
    C.text(HX0 - 2.0, EB0 + EH / 2 + EH / 2 * v / gmax, lab, ha="right", fontsize=4.9, color=MUTED)
for grp, lab, col in ((DOWN, f"Lower in PD ({len(DOWN)})", DNC), (UP, f"Higher in PD ({len(UP)})", UPC)):
    xa, xb = gx[grp[0]] - cw / 2, gx[grp[-1]] + cw / 2
    M.add_patch(Rectangle((xa + 0.2, EB0 + EH + 1.4), xb - xa - 0.4, 0.7, facecolor=col, lw=0))
    C.text((xa + xb) / 2, EB0 + EH + 3.2, lab, ha="center", va="bottom", fontsize=5.8, color=col, fontweight="bold")
# heatmap
RH = 3.7
SOXMAP = LinearSegmentedColormap.from_list("sox", ["#F3F5F7", "#9AAABB", "#4C6583", "#1E3350"])
CALMAP = LinearSegmentedColormap.from_list("cal", ["#F7F3F3", "#C3A09E", "#8C4B52", "#551C28"])
ZMAX = 30.0
hy = {}
y_ = EB0 - 2.2
for i, st in enumerate(SUBTYPES):
    if i == 4:
        y_ -= 1.4
    hy[st] = y_ - RH / 2; y_ -= RH
for st in SUBTYPES:
    cmap = SOXMAP if st.startswith("SOX6") else CALMAP
    yc = hy[st]
    C.text(HX0 - 2.0, yc, st.replace("_", " "), ha="right", fontsize=5.6, fontweight="bold" if st == "SOX6_AGTR1" else "normal")
    for g in genes:
        z = ZP.loc[g, st]
        M.add_patch(Rectangle((gx[g] - cw / 2, yc - RH / 2), cw, RH, facecolor=cmap(min(z, ZMAX) / ZMAX) if z > 0 else "#FFFFFF",
                              edgecolor="white", lw=0.5))
        if z >= 5:
            C.text(gx[g], yc, f"{z:.0f}", ha="center", fontsize=4.4, color="white" if z >= 13 else INK)
C.text(8.5, hy["SOX6_AGTR1"], "lost in PD", ha="left", fontsize=5.0, color=MUTED, fontstyle="italic")
for grp, sts, col in (("SOX6", SUBTYPES[:4], DNC), ("CALB1", SUBTYPES[4:], UPC)):
    ya, yb = hy[sts[0]] + RH / 2, hy[sts[-1]] - RH / 2
    M.plot([5.0, 5.0], [yb + 0.2, ya - 0.2], color=col, lw=1.4, solid_capstyle="butt")
    C.text(3.3, (ya + yb) / 2, grp, ha="center", rotation=90, fontsize=5.6, color=col, fontweight="bold")
# lineage score strip and gene names below
LYS = hy[SUBTYPES[-1]] - RH / 2 - 2.6
lmap = dict(zip(PLN.name, PLN.lineage))
LINMAP = LinearSegmentedColormap.from_list("lin", ["#1E3350", "#4C6583", "#9AAABB", "#EEECE7", "#C3A09E", "#8C4B52", "#551C28"])
for g in genes:
    v = lmap[g]
    M.add_patch(Rectangle((gx[g] - cw / 2, LYS - 1.3), cw, 2.6, facecolor=LINMAP(0.5 + 0.5 * np.clip(tr(v) / tr(16), -1, 1)),
                          edgecolor="white", lw=0.5))
C.text(HX0 - 2.0, LYS, "Lineage score", ha="right", fontsize=5.6)
for g in genes:
    C.text(gx[g], LYS - 2.2, g, ha="center", va="top", rotation=90, fontsize=5.6, fontstyle="italic", color=DNC if g in DOWN else UPC)
# colour keys
KY0 = 3.2
for k, (cmap, lab) in enumerate(((SOXMAP, "SOX6 subtypes"), (CALMAP, "CALB1 subtypes"))):
    cax = C.ax(16.0 + k * 40.0, KY0 - 0.8, 16.0, 1.6)
    cax.imshow(np.linspace(0, 1, 256)[None, :], aspect="auto", cmap=cmap); cax.set_xticks([]); cax.set_yticks([])
    for s_ in cax.spines.values():
        s_.set_linewidth(0.3); s_.set_color("#B7BDC4")
    C.text(15.2 + k * 40.0, KY0, "0", ha="right", fontsize=5.2, color=MUTED)
    C.text(32.8 + k * 40.0, KY0, f"{ZMAX:.0f}+  {lab}", ha="left", fontsize=5.2, color=MUTED)
C.text(4.0, KY0, "Marker z", ha="left", fontsize=5.4)
cax = C.ax(110.0, KY0 - 0.8, 16.0, 1.6)
cax.imshow(np.linspace(0, 1, 256)[None, :], aspect="auto", cmap=LINMAP); cax.set_xticks([]); cax.set_yticks([])
for s_ in cax.spines.values():
    s_.set_linewidth(0.3); s_.set_color("#B7BDC4")
C.text(109.2, KY0, "SOX6", ha="right", fontsize=5.2, color=DNC)
C.text(126.8, KY0, "CALB1   lineage score", ha="left", fontsize=5.2, color=MUTED)
C.text(181.0, KY0, "numbers: marker z >= 5", ha="right", fontsize=5.2, color=MUTED)
C.save("Figure08_panel_subtypes")


## Figure 9 - pathways (183 x 195 mm)

In [ ]:
UPC, DNC, ALLC = PANEL_C, CORE_C, "#3E454D"
C = Canvas(183.0, 195.0); M = C.M
def chip(x, y_, lib):
    M.add_patch(Rectangle((x, y_ - 1.35), 10.2, 2.7, facecolor="#E9ECEF", edgecolor="none", zorder=2))
    C.text(x + 5.1, y_, LIBTAG.get(lib, lib), ha="center", fontsize=4.6, color="#3E454D", zorder=3)

# ======================= a: over-representation with the genes behind each pathway =======================
C.letter(2.0, 191.0, "a", "Pathway over-representation of the lower-, higher- and all panel genes")
LISTS = [("up", UPC, f"Higher in PD ({len(UP)} genes)"), ("down", DNC, f"Lower in PD ({len(DOWN)} genes)"), ("all", ALLC, "All 30 genes")]
ROWS = []
for lst, col, lab in LISTS:
    ROWS.append(("header", lst, col, lab))
    for r in ORT[ORT.list == lst].nsmallest(5, "p").itertuples():
        ROWS.append(("term", lst, col, r))
GENES_A = []
for kind, lst, col, r in ROWS:
    if kind == "term":
        for g in r.genes.split(", "):
            if nice(g) not in GENES_A:
                GENES_A.append(nice(g))
GENES_A = [g for g in DOWN + UP if g in GENES_A]
TX, AX0, AX1, FDX, FWX, GX0, GX1 = 15.0, 78.0, 104.0, 110.0, 119.5, 126.0, 181.0
gcw = (GX1 - GX0) / len(GENES_A)
GXP = {g: GX0 + (i + 0.5) * gcw for i, g in enumerate(GENES_A)}
HY = 175.5
for x, t, ha in ((3.0, "Library", "left"), (TX, "Pathway", "left"), ((AX0 + AX1) / 2, "$-\\log_{10}$ P", "center"), (FDX, "FDR", "center"),
                 (FWX, "Family-\nwise P", "center")):
    C.text(x, HY, t, ha=ha, fontsize=5.7, linespacing=1.0)
for g in GENES_A:
    C.text(GXP[g], HY - 1.6, g, ha="center", va="bottom", rotation=90, fontsize=5.4, fontstyle="italic",
           color=DNC if DIRN[g] == "down" else UPC)
C.text((GX0 + GX1) / 2, 187.6, "Panel genes in the pathway", ha="center", fontsize=5.7)
C.rule(2.0, 181.0, HY - 2.6 - 0.0, lw=0.5, color=INK)
y_ = HY - 5.4
YPOS, GROUP_SPAN = [], {}
for kind, lst, col, r in ROWS:
    if kind == "header":
        C.text(TX, y_, r, ha="left", fontsize=5.8, color=col, fontweight="bold")
        GROUP_SPAN[lst] = [y_ - 2.6, None]; y_ -= 4.4
    else:
        YPOS.append((y_, lst, col, r)); GROUP_SPAN[lst][1] = y_ - 2.6; y_ -= 5.3
axA = C.ax(AX0, y_ + 2.0, AX1 - AX0, HY - 3.2 - (y_ + 2.0)); axA.set_ylim(y_ + 2.0, HY - 3.2); axA.set_xlim(0, 4.6)
axA.spines["left"].set_visible(False); axA.set_yticks([]); axA.patch.set_alpha(0)
axA.set_xticks([0, 1, 2, 3, 4]); axA.tick_params(axis="x", labelsize=5.6)
for lst, (ya, yb) in GROUP_SPAN.items():
    t = -np.log10(SMY["ora_calibration"][lst]["p_fwer05_random"])
    axA.plot([t, t], [yb + 0.6, ya], color="#8C9299", lw=0.7, ls=(0, (2.2, 1.6)), zorder=1)
for i, (yy, lst, col, r) in enumerate(YPOS):
    if i % 2 == 0:
        M.add_patch(Rectangle((2.0, yy - 2.6), 179.0, 5.2, facecolor="#F3F5F7", lw=0, zorder=0))
    chip(3.0, yy, r.library)
    lines = wrap(term_name(r.term), 58)
    C.text(TX, yy, "\n".join(lines), ha="left", fontsize=5.3, linespacing=1.02)
    x = -np.log10(r.p)
    axA.plot([0, x], [yy, yy], color="#C9CED4", lw=0.6, zorder=2)
    axA.scatter([x], [yy], s=8 + 8 * r.hits, color=col, zorder=3, linewidth=0)
    C.text(FDX, yy, f"{r.q_BH:.2f}", ha="center", fontsize=5.4)
    C.text(FWX, yy, f"{r.fwer_random:.3f}" if r.fwer_random < 0.1 else f"{r.fwer_random:.2f}", ha="center", fontsize=5.4)
    members = {nice(g) for g in r.genes.split(", ")}
    for g in GENES_A:
        if g in members:
            M.scatter([GXP[g]], [yy], s=15, color=DNC if DIRN[g] == "down" else UPC, lw=0, zorder=3)
        else:
            M.scatter([GXP[g]], [yy], s=1.6, color="#C9CED4", lw=0, zorder=2)
C.text(3.0, y_ - 3.2, f"Dashed lines: the P value that random gene sets of the same size reach, anywhere among the "
       f"{SMY['ora_calibration']['all']['terms']:,} pathways tested, in 5% of draws (family-wise 5% threshold). Dot area: number of panel genes.",
       ha="left", fontsize=5.3, color=MUTED)
BOT_A = y_ - 5.6
C.rule(2.0, 181.0, BOT_A, lw=0.35, color="#8C9299")

# ======================= b: do the pathway partners move with the panel gene? =======================
C.letter(2.0, BOT_A - 4.6, "b", "Pathway partners of the panel genes (label-shuffle test)")
NB = NBT.head(8)
TX2, AGX, BX0, BX1, PX, QX, NX = 15.0, 70.0, 86.0, 110.0, 116.0, 125.0, 131.0
HY2 = BOT_A - 10.6
for x, t, ha in ((3.0, "Library", "left"), (TX2, "Pathway", "left"), (AGX, "Panel gene", "left"), ((BX0 + BX1) / 2, "Partner shift (g)", "center"),
                 (PX, "P", "center"), (QX, "FDR", "center"), (NX, "Partners moving most with it", "left")):
    C.text(x, HY2, t, ha=ha, fontsize=5.7)
C.rule(2.0, 181.0, HY2 - 2.2, lw=0.5, color=INK)
y2 = HY2 - 5.4
YB = []
for i, r in enumerate(NB.itertuples()):
    if i % 2 == 0:
        M.add_patch(Rectangle((2.0, y2 - 2.6), 179.0, 5.2, facecolor="#F3F5F7", lw=0, zorder=0))
    col = UPC if r.direction == "up" else DNC
    chip(3.0, y2, r.library)
    C.text(TX2, y2, "\n".join(wrap(term_name(r.term), 50)), ha="left", fontsize=5.3, linespacing=1.02)
    C.text(AGX, y2, ", ".join(nice(a) for a in r.anchors.split(", ")), ha="left", fontsize=5.4, fontstyle="italic", color=col)
    C.text(PX, y2, f"{r.p:.3f}", ha="center", fontsize=5.4)
    C.text(QX, y2, f"{r.fdr:.2f}", ha="center", fontsize=5.4)
    C.text(NX, y2, ", ".join(r.top_neighbours.split(", ")[:6]), ha="left", fontsize=5.1, fontstyle="italic", color=MUTED)
    YB.append((y2, r, col)); y2 -= 5.3
axB = C.ax(BX0, y2 + 2.1, BX1 - BX0, HY2 - 2.6 - (y2 + 2.1)); axB.set_ylim(y2 + 2.1, HY2 - 2.6); axB.set_xlim(0, 0.3)
axB.spines["left"].set_visible(False); axB.set_yticks([]); axB.patch.set_alpha(0)
axB.set_xticks([0, 0.1, 0.2, 0.3]); axB.set_xticklabels(["0", "0.1", "0.2", "0.3"]); axB.tick_params(axis="x", labelsize=5.6)
for yy, r, col in YB:
    axB.plot([0, r.shift], [yy, yy], color="#C9CED4", lw=0.6, zorder=2)
    axB.scatter([r.shift], [yy], s=5 + 0.9 * r.neighbours, color=col, zorder=3, linewidth=0)
C.text(3.0, y2 - 4.6, f"The eight strongest of {len(NBT)} pathways holding a panel gene. Partner shift: mean PD effect of the pathway's other genes, "
       "in the panel gene's direction;", ha="left", fontsize=5.3, color=MUTED)
C.text(3.0, y2 - 7.2, "P from 5,000 label shuffles within study; none passes FDR or family-wise correction. Dot area: number of partner genes.",
       ha="left", fontsize=5.3, color=MUTED)
C.save("Figure09_panel_pathways")


## Figure S - panel gene summary (183 x 134 mm)

In [ ]:
from matplotlib.colors import LinearSegmentedColormap
UPC, DNC = PANEL_C, CORE_C
DIV = LinearSegmentedColormap.from_list("div", ["#1E3350", "#4C6583", "#9AAABB", "#EEECE7", "#C3A09E", "#8C4B52", "#551C28"])
SEQ = LinearSegmentedColormap.from_list("seq", ["#F4F5F6", "#C9CED4", "#8C9299", "#4A4F56", "#23282D"])
G = GTB.copy()
G["name_full"] = G["name"].fillna("")
G["grp"] = np.where(G.direction == "down", 0, 1)
G = G.sort_values(["grp", "hedges_g_discovery"], key=lambda s: s if s.name == "grp" else s.abs(), ascending=[True, False]).reset_index(drop=True)
C = Canvas(183.0, 134.0); M = C.M
C.text(2.0, 130.0, "Panel gene summary", ha="left", va="baseline", fontsize=7.2)
COLS = [  # key, header, kind, scale
    ("hedges_g_discovery", "PD effect\n(discovery g)", "div", 1.3),
    ("p_discovery", "Discovery\n$-\\log_{10}$ P", "seqlog", 4.0),
    ("single_gene_auc", "Single-gene\nAUC", "seq", (0.5, 0.8)),
    ("shap_rank", "SHAP\nrank", "rank", 30),
    ("boruta_fold_frequency", "Boruta fold\nfrequency", "seq", (0, 1)),
    ("g_external_neuron_adjusted", "Bulk nigra g\n(neuron-adj.)", "div", 0.7),
    ("dopamine_lineage_score", "Lineage\nscore", "lin", 16),
    ("top_subtype_marker", "Strongest\nsubtype marker", "text", None),
    ("pd_association_open_targets", "Open Targets\nPD score", "seq", (0, 0.6)),
    ("pd_genetic_association", "PD genetic\nevidence", "seq", (0, 0.8))]
X_SYM, X_NAME, X0 = 3.0, 17.0, 68.0
widths = [10.5, 10.5, 10.5, 9.0, 10.5, 11.5, 10.5, 17.0, 11.5, 11.0]
xs = np.cumsum([X0] + widths[:-1]); xc = xs + np.array(widths) / 2
HY = 122.5
C.text(X_SYM, HY, "Gene", ha="left", fontsize=5.8); C.text(X_NAME, HY, "Name", ha="left", fontsize=5.8)
for (k, h, kind, sc_), x in zip(COLS, xc):
    C.text(x, HY, h, ha="center", fontsize=5.4, linespacing=1.0)
C.rule(2.0, 181.0, HY - 3.4, lw=0.5, color=INK)
RH = 3.3
y_ = HY - 6.0
tr = lambda v: np.sign(v) * np.log10(1 + np.abs(v) / 2.0)
for grp, lab, col in ((0, f"Lower in PD ({len(DOWN)})", DNC), (1, f"Higher in PD ({len(UP)})", UPC)):
    C.text(X_SYM, y_, lab, ha="left", fontsize=5.8, color=col, fontweight="bold"); y_ -= 3.8
    sub = G[G.grp == grp]
    y_top = y_ + RH / 2
    for r in sub.itertuples():
        C.text(X_SYM + 1.4, y_, r.symbol, ha="left", fontsize=5.6, fontstyle="italic", color=col)
        C.text(X_NAME, y_, r.name_full[0].upper() + r.name_full[1:] if r.name_full else "", ha="left", fontsize=4.9, color="#3E454D")
        for (k, h, kind, sc_), x0, w in zip(COLS, xs, widths):
            v = getattr(r, k)
            face, txt, tcol = "#FFFFFF", "", INK
            if kind == "div":
                face = DIV(0.5 + 0.5 * np.clip(v / sc_, -1, 1)) if pd.notna(v) else "#FFFFFF"
                txt = f"{v:+.2f}" if pd.notna(v) else "n/a"
                tcol = "white" if pd.notna(v) and abs(v / sc_) > 0.55 else INK
                if k == "g_external_neuron_adjusted" and pd.notna(v) and np.sign(v) == np.sign(r.hedges_g_discovery):
                    txt += "*"
            elif kind == "seqlog":
                lv = -np.log10(v); face = SEQ(min(lv / sc_, 1)); txt = f"{lv:.1f}"; tcol = "white" if lv / sc_ > 0.6 else INK
            elif kind == "seq":
                a_, b_ = sc_; f = np.clip((v - a_) / (b_ - a_), 0, 1) if pd.notna(v) else 0
                face = SEQ(f); txt = (f"{v:.2f}" if pd.notna(v) else ""); tcol = "white" if f > 0.6 else INK
            elif kind == "rank":
                f = 1 - (v - 1) / (sc_ - 1); face = SEQ(f); txt = f"{int(v)}"; tcol = "white" if f > 0.6 else INK
            elif kind == "lin":
                face = DIV(0.5 + 0.5 * np.clip(tr(v) / tr(sc_), -1, 1)); txt = f"{v:+.1f}" if abs(v) >= 0.05 else "0"
                tcol = "white" if abs(tr(v) / tr(sc_)) > 0.55 else INK
            elif kind == "text":
                txt = str(v).replace("_", " ") if isinstance(v, str) and v else "-"
                tcol = DNC if txt.startswith("SOX6") else (UPC if txt.startswith("CALB1") else MUTED)
            M.add_patch(Rectangle((x0 + 0.25, y_ - RH / 2 + 0.2), w - 0.5, RH - 0.4, facecolor=face, edgecolor="none"))
            C.text(x0 + w / 2, y_, txt, ha="center", fontsize=4.9, color=tcol)
        y_ -= RH
    M.plot([2.0, 2.0], [y_ + RH / 2, y_top], color=col, lw=1.4, solid_capstyle="butt")
    y_ -= 1.2
C.rule(2.0, 181.0, y_ + 0.4, lw=0.5, color=INK)
C.text(2.0, y_ - 2.4, "Bulk nigra g: pooled over the 8 external cohorts after neuron content was regressed out; * same sign as discovery. "
       "Lineage score: CALB1 minus SOX6 subtype markers (Kamath et al. 2022).", ha="left", fontsize=5.2, color=MUTED)
C.text(2.0, y_ - 5.0, "Open Targets: association with Parkinson disease (overall and genetic evidence). SHAP rank 1 = most important "
       "gene in the panel forest; Boruta fold frequency = share of cross-validation folds selecting the gene.", ha="left", fontsize=5.2, color=MUTED)
C.save("FigureS_panel_gene_summary")
